<a href="https://colab.research.google.com/github/FreddyForero1988/skills-getting-started-with-github-copilot/blob/main/Transforma_textos_en_embeddings_Freddy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transforma textos en embeddings
# Procesamiento de Lenguaje Natural (608144)
# Especialización en Inteligencia Artificial - Universidad de Cundinamarca
# Autor: Freddy Alexander Forero Rojas
# Gestor de Conocimiento: Paul Alexander Diaz Montaña

La tokenización divide un texto en unidades llamadas tokens. En este laboratorio el `Tokenizer` de Keras trabaja a nivel de palabras y aprende un índice entero para cada palabra reconocida. Un embedding transforma después cada token en un vector denso de números reales.

Funciones que se ejecutan:

- Aprender un vocabulario con el `Tokenizer` de Keras.
- Construir la tabla de palabra, frecuencia e índice.
- Verificar la normalización (mayúsculas y puntuación).
- Convertir las oraciones en secuencias de índices.
- Mostrar por qué los índices no son vectores semánticos.
- Convertir los índices en vectores con una capa `Embedding`.
- (Opcional) Comparar con un modelo de embeddings preentrenado.

## 1. Importar las librerías

In [ ]:
import numpy as np # Operaciones numéricas.
import pandas as pd # Tablas (DataFrames) para mostrar resultados.
import tensorflow as tf # TensorFlow, backend de Keras.

from tensorflow.keras.preprocessing.text import Tokenizer # Clase que aprende un vocabulario y convierte textos en índices.
from tensorflow.keras.layers import Embedding # Capa que convierte índices en vectores densos entrenables.
from sklearn.metrics.pairwise import cosine_similarity # Similitud del coseno entre vectores.

### Análisis de la importación de librerías

Se importa `Tokenizer` desde `tensorflow.keras.preprocessing.text`. Esta clase permite aprender un vocabulario a partir de textos y posteriormente convertir textos en secuencias de índices. También se importa la capa `Embedding`, que se usa en la sección 7 para pasar de índices enteros a vectores densos, y `pandas` para presentar los resultados en tablas.

## 2. Definir las oraciones

In [ ]:
sentences = [
    'i love my dog',
    'I, love my cat',
    'You love my dog!'
]

print(f"Cantidad de oraciones: {len(sentences)}")

Cantidad de oraciones: 3


### Análisis de la definición de oraciones

La variable `sentences` contiene tres oraciones. El conjunto es pequeño y tiene una finalidad didáctica: permite observar cómo se construye un vocabulario a partir de palabras repetidas (`love`, `my`) y palabras que aparecen una sola vez (`cat`, `you`). Además, incluye una mayúscula (`I`) y signos de puntuación (`,` y `!`) para comprobar cómo los trata el tokenizer.

## 3. Aprender el vocabulario con el Tokenizer

In [ ]:
tokenizer = Tokenizer(num_words = 100) # Límite de vocabulario: 100 posiciones relevantes para las secuencias.
tokenizer.fit_on_texts(sentences) # Analiza las oraciones, identifica las palabras y construye el índice.
word_index = tokenizer.word_index # Diccionario palabra -> entero.
print(word_index)

{'love': 1, 'my': 2, 'i': 3, 'dog': 4, 'cat': 5, 'you': 6}


### Análisis del aprendizaje del vocabulario

- `Tokenizer(num_words=100)` establece un límite de vocabulario de 100 posiciones relevantes para las secuencias. En este ejemplo solo hay 6 palabras, por lo que el límite no restringe la información.
- `fit_on_texts(sentences)` analiza las oraciones, identifica las palabras y calcula la información necesaria para construir el índice de palabras.
- `tokenizer.word_index` devuelve un diccionario en el que cada palabra reconocida está asociada con un entero. Es la representación discreta aprendida.

El computador no necesita comprender el significado de las palabras para construir una primera representación numérica: el tokenizer identifica unidades léxicas, cuenta su aparición y las relaciona con identificadores enteros.

## 4. Tabla de palabra, frecuencia e índice

In [ ]:
tabla_vocabulario = pd.DataFrame({
    "Palabra": list(word_index.keys()),
    "Frecuencia": [tokenizer.word_counts[p] for p in word_index], # Cuántas veces aparece cada palabra.
    "Indice": list(word_index.values())
})

tabla_vocabulario

,Palabra,Frecuencia,Indice
0,love,3,1
1,my,3,2
2,i,2,3
3,dog,2,4
4,cat,1,5
5,you,1,6


### Análisis de la tabla del vocabulario

`love` y `my` aparecen en las tres oraciones y, por su mayor frecuencia, reciben los primeros índices (1 y 2). `i` y `dog` aparecen dos veces cada una. `cat` y `you` aparecen una sola vez y reciben los últimos índices.

**Nota sobre los empates:** cuando dos palabras tienen la misma frecuencia, Keras conserva el orden en que aparecieron por primera vez en el texto. Como `i` aparece antes que `dog` (en la primera oración), `i` recibe el índice 3 y `dog` el índice 4. Por eso la tabla del informe debe ajustarse: `dog` = 4 e `i` = 3, no al revés.

## 5. Verificar la normalización del tokenizer

In [ ]:
print("¿'I' aparece como token aparte?:", 'I' in word_index) # Falso: se normaliza a minúscula.
print("Frecuencia de 'i':", tokenizer.word_counts['i']) # 2: cuenta 'i' de la 1.ª oración e 'I' de la 2.ª.
print("¿Aparece ',' como token?:", ',' in word_index) # Falso: la puntuación se elimina.
print("¿Aparece '!' como token?:", '!' in word_index) # Falso.
print("Vocabulario total:", len(word_index), "palabras únicas")

¿'I' aparece como token aparte?: False
Frecuencia de 'i': 2
¿Aparece ',' como token?: False
¿Aparece '!' como token?: False
Vocabulario total: 6 palabras únicas


### Análisis de la normalización

La forma `I` de la segunda oración se normaliza a la misma representación que `i` de la primera, por lo que ambas contribuyen al mismo token `i` (frecuencia 2). Asimismo, la coma y el signo de exclamación no se convierten en tokens independientes, porque el tokenizer de Keras elimina la puntuación por defecto y pasa el texto a minúsculas.

## 6. Convertir las oraciones en secuencias de índices

In [ ]:
secuencias = tokenizer.texts_to_sequences(sentences) # Reemplaza cada palabra por su índice.

for oracion, secuencia in zip(sentences, secuencias):
    print(f"{oracion!r:22} -> {secuencia}")

'i love my dog'        -> [3, 1, 2, 4]
'I, love my cat'       -> [3, 1, 2, 5]
'You love my dog!'     -> [6, 1, 2, 4]


### Análisis de las secuencias

Cada oración queda convertida en una lista de enteros, que es el formato adecuado para alimentar etapas posteriores de un modelo. Esta representación es eficiente, pero cada número es solo un identificador: no contiene información sobre el significado de la palabra.

## 7. Los índices son categorías, no vectores semánticos

In [ ]:
love, my, dog = word_index['love'], word_index['my'], word_index['dog']

print(f"love={love}, my={my}, dog={dog}")
print("Distancia numérica love - my :", abs(love - my))
print("Distancia numérica love - dog:", abs(love - dog))

love=1, my=2, dog=4
Distancia numérica love - my : 1
Distancia numérica love - dog: 3


### Análisis

Que `love` tenga índice 1 y `my` índice 2 no indica que una palabra sea más cercana a la otra que a `dog`. La diferencia numérica (1 frente a 3) es un efecto del orden de frecuencia, no de significado. Por esta razón, utilizar directamente estos números como variables continuas introduciría una relación numérica artificial. La capa de embedding resuelve conceptualmente este problema al convertir cada índice en un vector entrenable de dimensión determinada.

## 8. Convertir los índices en vectores con una capa Embedding

In [ ]:
tf.keras.utils.set_random_seed(42) # Semilla para que los resultados sean reproducibles.

tamano_vocabulario = len(word_index) + 1 # +1 porque el índice 0 está reservado (relleno).
dimension_embedding = 8 # Dimensión de cada vector (pequeña, solo con fines didácticos).

capa_embedding = Embedding(input_dim=tamano_vocabulario, output_dim=dimension_embedding)

entrada = tf.constant(secuencias) # Matriz de índices (3 oraciones x 4 palabras, la 2.ª y 3.ª tienen la misma longitud).
vectores = capa_embedding(entrada)

print("Forma de la entrada (oraciones, palabras):", entrada.shape)
print("Forma de la salida (oraciones, palabras, dimensión):", vectores.shape)
print("Matriz de pesos del embedding:", capa_embedding.get_weights()[0].shape)
print("\nVector de 'love' (índice 1):")
print(capa_embedding.get_weights()[0][word_index['love']].round(4))

Forma de la entrada (oraciones, palabras): (3, 4)
Forma de la salida (oraciones, palabras, dimensión): (3, 4, 8)
Matriz de pesos del embedding: (7, 8)

Vector de 'love' (índice 1):
[ 0.0024  0.0131 -0.0414 -0.0054  0.0466 -0.003   0.0003  0.0266]


### Análisis de la capa Embedding

La capa `Embedding` es una tabla de búsqueda: tiene una fila por cada índice del vocabulario (7 filas: el índice 0 reservado más las 6 palabras) y una columna por cada dimensión (8). Al recibir un índice devuelve su fila, es decir, un vector denso de números reales. La salida tiene forma `(3, 4, 8)`: 3 oraciones, 4 palabras por oración y 8 valores por palabra.

**Importante:** estos vectores se inicializan de forma aleatoria. Mientras la capa no se entrene dentro de un modelo, no codifican ningún significado. El embedding puede aprenderse desde cero durante el entrenamiento o provenir de un modelo previamente entrenado, como se muestra en la sección 10.

## 9. Limitación del conjunto de datos

In [ ]:
print("Oraciones:", len(sentences))
print("Palabras únicas:", len(word_index))

similitud = cosine_similarity(capa_embedding.get_weights()[0][1:]) # Se omite el índice 0.
palabras = list(word_index.keys())
pd.DataFrame(similitud, index=palabras, columns=palabras).round(2)

Oraciones: 3
Palabras únicas: 6


,love,my,i,dog,cat,you
love,1.00,-0.45,0.24,0.50,0.44,-0.23
my,-0.45,1.00,-0.53,0.24,0.10,0.25
i,0.24,-0.53,1.00,-0.00,-0.43,-0.06
dog,0.50,0.24,-0.00,1.00,0.66,0.33
cat,0.44,0.10,-0.43,0.66,1.00,-0.15
you,-0.23,0.25,-0.06,0.33,-0.15,1.00


### Análisis de la limitación

El conjunto solo contiene tres oraciones y seis palabras únicas. Con tan pocos ejemplos no es posible aprender relaciones semánticas robustas. La matriz de similitud del coseno entre los vectores aleatorios sin entrenar lo evidencia: los valores no reflejan ninguna relación real entre las palabras (por ejemplo, entre `dog` y `cat`). Para que palabras usadas en contextos semejantes queden cercanas se necesita entrenar la capa con muchos más datos o usar embeddings preentrenados.

## 10. (Opcional) Embeddings preentrenados con sentence-transformers

Esta sección muestra el contraste con un modelo ya entrenado, basado en el enfoque del cuaderno de referencia. Requiere instalar `sentence-transformers` (en Colab: `!pip install -q sentence-transformers`).

In [ ]:
# !pip install -q sentence-transformers  # Descomentar en Colab
from sentence_transformers import SentenceTransformer

nombre_modelo = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
modelo = SentenceTransformer(nombre_modelo)

emb_pre = modelo.encode(sentences, convert_to_numpy=True, normalize_embeddings=True)
print("Forma de la matriz:", emb_pre.shape) # (3, 384)

sim_pre = cosine_similarity(emb_pre)
pd.DataFrame(sim_pre, index=[f"Oración {i+1}" for i in range(3)],
             columns=[f"Oración {i+1}" for i in range(3)]).round(4)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Forma de la matriz: (3, 384)


,Oración 1,Oración 2,Oración 3
Oración 1,1.0000,0.5493,0.9269
Oración 2,0.5493,1.0000,0.5088
Oración 3,0.9269,0.5088,1.0000


### Análisis

Aquí cada oración completa se convierte en un vector denso de 384 dimensiones aprendido previamente con grandes volúmenes de texto. A diferencia de los índices del `Tokenizer`, la similitud del coseno entre estos vectores sí refleja cercanía de significado: las oraciones 1 y 2 (`i love my dog` / `I, love my cat`) deberían resultar más parecidas entre sí que con otras frases sin relación.

## Conclusiones

- La transformación de lenguaje natural a datos procesables por un modelo requiere una representación numérica de los tokens.
- El código utiliza el tokenizer de Keras para aprender un vocabulario y asignar índices enteros a las palabras presentes en las oraciones.
- La normalización del tokenizer permite que las variantes de mayúsculas y minúsculas se traten como la misma palabra, mientras que determinados signos de puntuación no se consideran tokens independientes.
- Los índices generados no son embeddings: son identificadores discretos. Para obtener una representación vectorial densa se necesita una capa de embedding o un mecanismo equivalente.
- Con solo tres oraciones y seis palabras únicas no es posible aprender relaciones semánticas robustas.